# 01 — Data Engineering & ResNet Training

> **Google Colab:** Go to `Runtime → Change runtime type → GPU` for faster training.

This notebook:
1. Fetches the **UCI Connect-4 Dataset** (67,557 board positions)
2. Parses text encodings (`'b'`, `'x'`, `'o'`) into **one-hot 3×6×7 tensors**
3. Trains a **ResNet** (5 residual blocks, dual policy+value heads)
4. Saves trained weights to Google Drive for use in the Pygame application

In [ ]:
!pip install -q ucimlrepo

In [ ]:
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

# Board constants (must match app/engine.py)
ROWS = 6
COLS = 7

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')

## 1. Fetch the UCI Connect-4 Dataset

In [ ]:
from ucimlrepo import fetch_ucirepo

dataset = fetch_ucirepo(id=26)
X = dataset.data.features   # DataFrame: 67557 rows × 42 columns
y = dataset.data.targets    # DataFrame: 67557 rows × 1 column

print(f'Features shape: {X.shape}')
print(f'Targets shape:  {y.shape}')
print(f'Target values:  {y.iloc[:, 0].value_counts().to_dict()}')
print(f'\nFeature columns: {list(X.columns[:6])} ... {list(X.columns[-6:])}')
print(f'Sample row:\n{X.iloc[0]}')

## 2. Parse Board Positions into 3×6×7 Tensors

**UCI column naming:** `a1`..`g6` where:
- Letter `a`–`g` → board column 0–6
- Number `1`–`6` → board row from **bottom** (row 1 = bottom = our array row 5)

**Encoding:** `'b'` → empty, `'x'` → player 1, `'o'` → player 2

**Tensor channels** (from player `x`'s perspective, since labels are from `x`'s perspective):
- Channel 0: player `x`'s pieces (current player)
- Channel 1: player `o`'s pieces (opponent)
- Channel 2: empty cells

In [ ]:
def parse_uci_to_tensors(X_df):
    """Convert UCI Connect-4 DataFrame into one-hot 3×6×7 numpy tensors.
    
    Args:
        X_df: DataFrame with 42 columns named a1..g6, values 'b'/'x'/'o'
    
    Returns:
        np.ndarray of shape (N, 3, 6, 7) with float32 values
    """
    n_samples = len(X_df)
    tensors = np.zeros((n_samples, 3, ROWS, COLS), dtype=np.float32)
    
    # Mapping: UCI column letter -> board column index
    col_map = {'a': 0, 'b': 1, 'c': 2, 'd': 3, 'e': 4, 'f': 5, 'g': 6}
    
    for col_name in X_df.columns:
        # Parse column name like 'c3' -> board col 2, UCI row 3
        board_col = col_map[col_name[0]]
        uci_row = int(col_name[1])          # 1-6, bottom-up
        board_row = ROWS - uci_row           # Convert to top-down (row 0 = top)
        
        values = X_df[col_name].values
        
        # Channel 0: player x's pieces (current player perspective)
        tensors[:, 0, board_row, board_col] = (values == 'x').astype(np.float32)
        # Channel 1: player o's pieces (opponent)
        tensors[:, 1, board_row, board_col] = (values == 'o').astype(np.float32)
        # Channel 2: empty cells
        tensors[:, 2, board_row, board_col] = (values == 'b').astype(np.float32)
    
    return tensors


def encode_labels(y_df):
    """Encode outcome labels as float values.
    
    win  -> +1.0  (player x wins)
    loss -> -1.0  (player x loses)
    draw ->  0.0
    """
    label_map = {'win': 1.0, 'loss': -1.0, 'draw': 0.0}
    return np.array([label_map[v] for v in y_df.iloc[:, 0].values], dtype=np.float32)


# Parse
X_tensors = parse_uci_to_tensors(X)
y_values = encode_labels(y)

print(f'Tensor shape: {X_tensors.shape}')   # (67557, 3, 6, 7)
print(f'Labels shape: {y_values.shape}')     # (67557,)
print(f'Label distribution: +1={np.sum(y_values==1)}, -1={np.sum(y_values==-1)}, 0={np.sum(y_values==0)}')

# Sanity check: each cell should have exactly one channel = 1
channel_sums = X_tensors.sum(axis=1)  # Sum over channels -> (N, 6, 7)
assert np.allclose(channel_sums, 1.0), 'One-hot encoding error: channels do not sum to 1'
print('\nOne-hot encoding verified: all cells sum to 1.0 across channels')

## 3. Visualize a Sample Board Position

In [ ]:
def visualize_board(tensor, label=None, idx=0):
    """Display a board tensor as a color grid."""
    fig, ax = plt.subplots(1, 1, figsize=(5, 4))
    
    # Reconstruct board: 0=empty, 1=player_x, 2=player_o
    board = np.zeros((ROWS, COLS))
    board[tensor[0] == 1] = 1   # Player X
    board[tensor[1] == 1] = -1  # Player O
    
    ax.imshow(board, cmap='RdYlBu', vmin=-1, vmax=1, aspect='equal')
    ax.set_xticks(range(COLS))
    ax.set_yticks(range(ROWS))
    ax.set_xticklabels([f'Col {i}' for i in range(COLS)])
    ax.set_yticklabels([f'Row {i}' for i in range(ROWS)])
    ax.set_title(f'Sample {idx}' + (f' — Outcome: {label:+.0f}' if label is not None else ''))
    
    # Add text annotations
    for r in range(ROWS):
        for c in range(COLS):
            if tensor[0, r, c] == 1:
                ax.text(c, r, 'X', ha='center', va='center', fontsize=14, fontweight='bold')
            elif tensor[1, r, c] == 1:
                ax.text(c, r, 'O', ha='center', va='center', fontsize=14, fontweight='bold')
    
    plt.tight_layout()
    plt.show()

# Show a few samples
for i in [0, 100, 1000]:
    visualize_board(X_tensors[i], y_values[i], idx=i)

## 4. Train/Validation Split & DataLoaders

In [ ]:
# Stratified 80/20 split
X_train, X_val, y_train, y_val = train_test_split(
    X_tensors, y_values, test_size=0.2, random_state=42, stratify=y_values
)

print(f'Training set:   {X_train.shape[0]} samples')
print(f'Validation set: {X_val.shape[0]} samples')

# Create DataLoaders
BATCH_SIZE = 256

train_dataset = TensorDataset(
    torch.tensor(X_train),
    torch.tensor(y_train).unsqueeze(1)  # Shape: (N, 1)
)
val_dataset = TensorDataset(
    torch.tensor(X_val),
    torch.tensor(y_val).unsqueeze(1)
)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE)

## 5. Model Definition

The ResNet architecture is defined inline so this notebook is self-contained
on Google Colab. The same architecture is used in `app/agents/resnet_agent.py`
for Pygame inference — keep them in sync if you modify either.

In [ ]:
class ResidualBlock(nn.Module):
    """Single residual block: Conv->BN->ReLU->Conv->BN + skip -> ReLU."""

    def __init__(self, num_filters):
        super().__init__()
        self.conv1 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn1 = nn.BatchNorm2d(num_filters)
        self.conv2 = nn.Conv2d(num_filters, num_filters, 3, padding=1, bias=False)
        self.bn2 = nn.BatchNorm2d(num_filters)

    def forward(self, x):
        residual = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        out = out + residual
        return F.relu(out)


class Connect4ResNet(nn.Module):
    """ResNet for Connect-4 with dual heads (policy + value).

    Architecture:
        Input: 3x6x7 (one-hot encoded board)
        Initial Conv: 3 -> num_filters, 3x3
        Residual Blocks: num_blocks x ResidualBlock
        Policy Head: 1x1 conv -> flatten -> FC -> 7 outputs
        Value Head: 1x1 conv -> flatten -> FC -> FC -> 1 output (tanh)
    """

    def __init__(self, num_blocks=5, num_filters=128):
        super().__init__()
        self.initial_conv = nn.Conv2d(3, num_filters, 3, padding=1, bias=False)
        self.initial_bn = nn.BatchNorm2d(num_filters)
        self.blocks = nn.Sequential(
            *[ResidualBlock(num_filters) for _ in range(num_blocks)]
        )

        # Policy head: predicts move probabilities for each column
        self.policy_conv = nn.Conv2d(num_filters, 2, 1, bias=False)
        self.policy_bn = nn.BatchNorm2d(2)
        self.policy_fc = nn.Linear(2 * ROWS * COLS, COLS)

        # Value head: predicts board evaluation in [-1, 1]
        self.value_conv = nn.Conv2d(num_filters, 1, 1, bias=False)
        self.value_bn = nn.BatchNorm2d(1)
        self.value_fc1 = nn.Linear(ROWS * COLS, 64)
        self.value_fc2 = nn.Linear(64, 1)

    def forward(self, x):
        # Shared trunk
        out = F.relu(self.initial_bn(self.initial_conv(x)))
        out = self.blocks(out)

        # Policy head
        p = F.relu(self.policy_bn(self.policy_conv(out)))
        p = p.view(p.size(0), -1)
        p = self.policy_fc(p)  # Raw logits (shape: [batch, 7])

        # Value head
        v = F.relu(self.value_bn(self.value_conv(out)))
        v = v.view(v.size(0), -1)
        v = F.relu(self.value_fc1(v))
        v = torch.tanh(self.value_fc2(v))  # Range [-1, 1]

        return p, v

In [ ]:
model = Connect4ResNet(num_blocks=5, num_filters=128).to(device)

# Count parameters
total_params = sum(p.numel() for p in model.parameters())
trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
print(f'Total parameters:     {total_params:,}')
print(f'Trainable parameters: {trainable_params:,}')

# Quick forward pass test
dummy = torch.randn(2, 3, 6, 7).to(device)
policy, value = model(dummy)
print(f'\nPolicy output shape: {policy.shape}')  # (2, 7)
print(f'Value output shape:  {value.shape}')      # (2, 1)
print(f'Value range: [{value.min().item():.3f}, {value.max().item():.3f}]')

## 6. Training Loop

We train the **value head** using MSE loss against the game outcome labels.
The UCI dataset doesn't include move labels, so the policy head is not
directly trained here — instead, the Pygame agent uses value-based
1-ply lookahead (evaluate all 7 next positions) for move selection.

In [ ]:
EPOCHS = 25
LR = 0.001
WEIGHT_DECAY = 1e-4

optimizer = torch.optim.Adam(model.parameters(), lr=LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.StepLR(optimizer, step_size=10, gamma=0.5)
criterion = nn.MSELoss()

train_losses = []
val_losses = []

for epoch in range(EPOCHS):
    # --- Training ---
    model.train()
    epoch_train_loss = 0.0
    n_batches = 0
    
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)
        
        optimizer.zero_grad()
        _, value_pred = model(X_batch)
        loss = criterion(value_pred, y_batch)
        loss.backward()
        optimizer.step()
        
        epoch_train_loss += loss.item()
        n_batches += 1
    
    avg_train_loss = epoch_train_loss / n_batches
    train_losses.append(avg_train_loss)
    
    # --- Validation ---
    model.eval()
    epoch_val_loss = 0.0
    n_val_batches = 0
    
    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            _, value_pred = model(X_batch)
            loss = criterion(value_pred, y_batch)
            epoch_val_loss += loss.item()
            n_val_batches += 1
    
    avg_val_loss = epoch_val_loss / n_val_batches
    val_losses.append(avg_val_loss)
    
    scheduler.step()
    
    if (epoch + 1) % 5 == 0 or epoch == 0:
        print(f'Epoch {epoch+1:3d}/{EPOCHS} | '
              f'Train Loss: {avg_train_loss:.4f} | '
              f'Val Loss: {avg_val_loss:.4f} | '
              f'LR: {scheduler.get_last_lr()[0]:.6f}')

## 7. Training Loss Curves

In [ ]:
plt.figure(figsize=(8, 5))
plt.plot(range(1, EPOCHS+1), train_losses, label='Train Loss', linewidth=2)
plt.plot(range(1, EPOCHS+1), val_losses, label='Val Loss', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('MSE Loss')
plt.title('ResNet Training — Value Head Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 8. Save Trained Weights

Saves to Google Drive for persistence. After training, download
`connect4_resnet.pt` and place it in `app/models/` to use with
the Pygame application.

In [ ]:
import os
from google.colab import drive

# Mount Google Drive for persistent storage
drive.mount('/content/drive')

# Save to Google Drive
drive_path = '/content/drive/MyDrive/connect4_resnet.pt'
torch.save(model.state_dict(), drive_path)
print(f'Model saved to Google Drive: {drive_path}')
print(f'File size: {os.path.getsize(drive_path) / 1024 / 1024:.1f} MB')

# Also save locally for immediate use in this session
local_path = '/content/connect4_resnet.pt'
torch.save(model.state_dict(), local_path)
print(f'Model also saved locally: {local_path}')
print(f'\nTo use with the Pygame app, download the .pt file and place it in:')
print(f'  app/models/connect4_resnet.pt')

## 9. Quick Sanity Check

Verify the saved model loads correctly and produces sensible outputs.

In [ ]:
# Reload the model from disk
test_model = Connect4ResNet(num_blocks=5, num_filters=128)
test_model.load_state_dict(torch.load(local_path, map_location='cpu', weights_only=True))
test_model.eval()

# Evaluate on a few validation samples
with torch.no_grad():
    sample_X = torch.tensor(X_val[:10])
    sample_y = y_val[:10]
    _, predictions = test_model(sample_X)
    predictions = predictions.squeeze().numpy()

print('Sample predictions vs actual labels:')
print(f'{"Predicted":>12s} {"Actual":>8s} {"Correct Sign":>14s}')
print('-' * 36)
correct = 0
for pred, actual in zip(predictions, sample_y):
    sign_match = (pred * actual > 0) or (abs(actual) < 0.01)
    correct += int(sign_match)
    print(f'{pred:+12.4f} {actual:+8.1f} {"Yes" if sign_match else "No":>14s}')

print(f'\nSign accuracy on sample: {correct}/{len(sample_y)}')

# Full validation accuracy (sign agreement)
with torch.no_grad():
    all_preds = []
    for X_batch, _ in val_loader:
        _, v = test_model(X_batch)
        all_preds.append(v.squeeze().numpy())
    all_preds = np.concatenate(all_preds)

sign_correct = np.sum((all_preds * y_val > 0) | (np.abs(y_val) < 0.01))
print(f'\nValidation sign accuracy: {sign_correct}/{len(y_val)} ({100*sign_correct/len(y_val):.1f}%)')